# 04_audit_and_schema

This notebook:
- Loads `Dataset/combined_dataset_standardised.csv`
- Creates stable `sample_id`
- Exports:
  - `Dataset/combined_stage1.csv` (image-only)
  - `Dataset/combined_stage2.csv` (metadata-only, leakage-free)
  - `Dataset/schema.json` (Stage2 contract)

**Assumptions:**
- Required columns exist: `is_malignant`, `img_path`, `dataset_id`, `patient_global`
- Standardised combined dataset already created by `02_dataset_standardarising.ipynb`


In [17]:
from utils.paths import dataset_dir
import pandas as pd
import json

DATASET_DIR = dataset_dir()

IN_PATH = DATASET_DIR / "combined_dataset_standardised.csv"
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", len(df.columns))
df.head(3)

Loaded: /Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/combined_dataset_standardised.csv
Shape: (5659, 42)
Columns: 42


,is_malignant,diagnosis,label_source,dataset_id,patient_global,img_path,age,sex,fitzpatrick,lesion_size_mm,...,distance,is_control,melanoma_flag,pathology_report,anatomical_site_clean,cli_impres__group,cli_impres__2_group,cli_impres__3_group,race_group,distance_group
0,0.0,nevus,pathology,A,A_PAT_1516,pad_images/PAT_1516_1765_530.png,8,unknown,0,-1.0,...,-1,unknown,unknown,unknown,arm,other_unclassified,other_unclassified,other_unclassified,unknown,NaN
1,1.0,bcc,pathology,A,A_PAT_46,pad_images/PAT_46_881_939.png,55,female,3,6.0,...,-1,unknown,unknown,unknown,neck,other_unclassified,other_unclassified,other_unclassified,unknown,NaN
2,1.0,ak,pathology,A,A_PAT_1545,pad_images/PAT_1545_1867_547.png,77,unknown,0,-1.0,...,-1,unknown,unknown,unknown,face,other_unclassified,other_unclassified,other_unclassified,unknown,NaN


In [18]:
REQUIRED = [
    "is_malignant", "img_path", "dataset_id", "patient_global"
]

missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Label balance (is_malignant):")
print(df["is_malignant"].value_counts(dropna=False))

print("\nExample image paths:")
print(df["img_path"].head())


Label balance (is_malignant):
is_malignant
1.0    3402
0.0    2257
Name: count, dtype: int64

Example image paths:
0    pad_images/PAT_1516_1765_530.png
1       pad_images/PAT_46_881_939.png
2    pad_images/PAT_1545_1867_547.png
3    pad_images/PAT_1989_4061_934.png
4     pad_images/PAT_684_1302_588.png
Name: img_path, dtype: object


## Define Stage2 feature schema

We explicitly define:
- **Stage2 Features** (allowed metadata)
- **Leakage / post-diagnostic / dataset-fingerprint columns** (excluded)

If a Stage2 feature is missing from the dataframe, it will be skipped automatically.


In [19]:
LABEL_COL = "is_malignant"
IMG_COL = "img_path"
ID_COLS = ["dataset_id", "patient_global"]

# Leakage / post-diagnostic / dataset-fingerprint columns to exclude from Stage2
LEAKAGE_COLS = [
    "clinical_impression", "clinical_impression_2", "clinical_impression_3",
    "cli_impres_group", "cli_impres_2_group", "cli_impres_3_group",
    "melanoma_flag", "pathology_report", "biopsed",
    "background_father", "background_mother",
    "race", "race_group",
    "distance", "distance_group",
    "is_control",
    # also exclude the multiclass label field from Stage2 by default
    "diagnosis",
]

# Stage2 features: safe + useful + present in your combined dataset
STAGE2_FEATURES = [
    # demographics
    "age",
    "sex",
    "fitzpatrick",

    # lesion characteristics
    "lesion_size_mm",
    "diameter_2",
    "anatomical_site_clean",

    # symptoms (optional but allowed)
    "bleed",
    "hurt",
    "itch",
    "changed",
    "grew",
    "elevation",

    # history & environment (optional)
    "smoking",
    "alcohol_consumption",
    "cancer_history",
    "skin_cancer_history",
    "pesticide",
    "has_piped_water",
    "has_sewage_system",
]

missing_stage2 = [c for c in STAGE2_FEATURES if c not in df.columns]
print("Missing Stage2 features (will be skipped if any):", missing_stage2)


Missing Stage2 features (will be skipped if any): []


## Create stable sample_id

This `sample_id` is used to align predictions across Stage1/Stage2/Stage3.


In [20]:
df["sample_id"] = (
    df["dataset_id"].astype(str) + "_" +
    df["patient_global"].astype(str) + "_" +
    df.index.astype(str)
)

print(df["sample_id"].head())
print("Unique sample_id:", df["sample_id"].nunique(), "/ rows:", len(df))


0    A_A_PAT_1516_0
1      A_A_PAT_46_1
2    A_A_PAT_1545_2
3    A_A_PAT_1989_3
4     A_A_PAT_684_4
Name: sample_id, dtype: object
Unique sample_id: 5659 / rows: 5659


## Export Stage1 dataset (image-only)

Outputs: `Dataset/combined_stage1.csv`


In [21]:
# 1) Create Stage1 dataframe (image-only columns)
stage1_cols = ["sample_id", IMG_COL, LABEL_COL] + ID_COLS
stage1 = df[stage1_cols].copy()

stage1 = stage1.rename(columns={
    IMG_COL: "image_path",
    LABEL_COL: "y",
})

if stage1["image_path"].isna().any():
    raise ValueError("Some image_path values are NaN.")

print("Stage1 dataframe created. Rows:", len(stage1))
stage1.head(3)


Stage1 dataframe created. Rows: 5659


,sample_id,image_path,y,dataset_id,patient_global
0,A_A_PAT_1516_0,pad_images/PAT_1516_1765_530.png,0.0,A,A_PAT_1516
1,A_A_PAT_46_1,pad_images/PAT_46_881_939.png,1.0,A,A_PAT_46
2,A_A_PAT_1545_2,pad_images/PAT_1545_1867_547.png,1.0,A,A_PAT_1545


In [22]:
# 2) Normalize MIDAS image paths to actual files on disk
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
MIDAS_DIR = PROJECT_ROOT / "midas_images"

def resolve_midas_path(rel_path: str) -> str:
    p = Path(rel_path)
    if (PROJECT_ROOT / p).exists():
        return str(p)

    base = p.with_suffix("")
    ext = p.suffix.lower()

    # Try extension swap
    for e in [".jpg", ".jpeg", ".JPG", ".JPEG"]:
        cand = base.with_suffix(e)
        if (PROJECT_ROOT / cand).exists():
            return str(cand)

    # Try _cropped variants
    if not base.name.endswith("_cropped"):
        for e in [ext, ".jpg", ".jpeg", ".JPG", ".JPEG"]:
            cand = Path(str(base) + "_cropped").with_suffix(e)
            if (PROJECT_ROOT / cand).exists():
                return str(cand)

    # Final fallback: case-insensitive lookup in MIDAS folder
    try:
        folder = PROJECT_ROOT / p.parent
        if folder.exists():
            lookup = {f.name.lower(): f.name for f in folder.iterdir() if f.is_file()}
            key = p.name.lower()
            if key in lookup:
                return str(p.parent / lookup[key])
    except Exception:
        pass

    return str(p)

mask = stage1["image_path"].str.contains("midas_images", na=False)
stage1.loc[mask, "image_path"] = stage1.loc[mask, "image_path"].apply(resolve_midas_path)

# Report and drop unresolved paths (rows where image file does not exist)
exists_mask = stage1["image_path"].apply(lambda p: (PROJECT_ROOT / p).exists())
unresolved = stage1[~exists_mask]
print("Unresolved image paths after MIDAS normalization:", len(unresolved))
stage1 = stage1[exists_mask].copy()
print("Dropped", len(unresolved), "rows. Stage1 rows now:", len(stage1))


Unresolved image paths after MIDAS normalization: 11
Dropped 11 rows. Stage1 rows now: 5648


In [23]:
# 3) Save Stage1 CSV (after path normalization so resolved paths are written)
OUT_STAGE1 = DATASET_DIR / "combined_stage1.csv"
stage1.to_csv(OUT_STAGE1, index=False)

print("Saved Stage1 CSV:", OUT_STAGE1)
stage1.head(3)

Saved Stage1 CSV: /Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/combined_stage1.csv


,sample_id,image_path,y,dataset_id,patient_global
0,A_A_PAT_1516_0,pad_images/PAT_1516_1765_530.png,0.0,A,A_PAT_1516
1,A_A_PAT_46_1,pad_images/PAT_46_881_939.png,1.0,A,A_PAT_46
2,A_A_PAT_1545_2,pad_images/PAT_1545_1867_547.png,1.0,A,A_PAT_1545


## Export Stage2 dataset (metadata-only, leakage-free)

Outputs: `Dataset/combined_stage2.csv`


In [24]:
stage2_features_existing = [c for c in STAGE2_FEATURES if c in df.columns]
stage2_cols = ["sample_id", LABEL_COL] + ID_COLS + stage2_features_existing
stage2 = df[stage2_cols].copy()

stage2 = stage2.rename(columns={LABEL_COL: "y"})
stage2 = stage2.drop(columns=[c for c in LEAKAGE_COLS if c in stage2.columns], errors="ignore")

OUT_STAGE2 = DATASET_DIR / "combined_stage2.csv"
stage2.to_csv(OUT_STAGE2, index=False)

print("Saved Stage2 CSV:", OUT_STAGE2)
print("Stage2 shape:", stage2.shape)
stage2.head(3)


Saved Stage2 CSV: /Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/combined_stage2.csv
Stage2 shape: (5659, 23)


,sample_id,y,dataset_id,patient_global,age,sex,fitzpatrick,lesion_size_mm,diameter_2,anatomical_site_clean,...,changed,grew,elevation,smoking,alcohol_consumption,cancer_history,skin_cancer_history,pesticide,has_piped_water,has_sewage_system
0,A_A_PAT_1516_0,0.0,A,A_PAT_1516,8,unknown,0,-1.0,-1.0,arm,...,false,false,false,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,A_A_PAT_46_1,1.0,A,A_PAT_46,55,female,3,6.0,5.0,neck,...,true,true,true,false,false,true,true,false,true,true
2,A_A_PAT_1545_2,1.0,A,A_PAT_1545,77,unknown,0,-1.0,-1.0,face,...,false,false,false,unknown,unknown,unknown,unknown,unknown,unknown,unknown


## Save schema.json

This is the Stage2 contract used by `Stage2/data/preprocessing.py`.


In [25]:
schema = {
    "task": "binary_classification",
    "label": "y",
    "image_column_stage1": "image_path",
    "id_columns": ["sample_id"] + ID_COLS,
    "patient_id": "patient_global",
    "stage2_features": stage2_features_existing,
    "excluded_columns": LEAKAGE_COLS,
    "source_input": str(IN_PATH.name),
    "generated_files": {
        "stage1_csv": "combined_stage1.csv",
        "stage2_csv": "combined_stage2.csv",
    },
}

SCHEMA_PATH = DATASET_DIR / "schema.json"
with open(SCHEMA_PATH, "w") as f:
    json.dump(schema, f, indent=2)

print("Saved schema:", SCHEMA_PATH)
schema


Saved schema: /Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/schema.json


{'task': 'binary_classification',
 'label': 'y',
 'image_column_stage1': 'image_path',
 'id_columns': ['sample_id', 'dataset_id', 'patient_global'],
 'patient_id': 'patient_global',
 'stage2_features': ['age',
  'sex',
  'fitzpatrick',
  'lesion_size_mm',
  'diameter_2',
  'anatomical_site_clean',
  'bleed',
  'hurt',
  'itch',
  'changed',
  'grew',
  'elevation',
  'smoking',
  'alcohol_consumption',
  'cancer_history',
  'skin_cancer_history',
  'pesticide',
  'has_piped_water',
  'has_sewage_system'],
 'excluded_columns': ['clinical_impression',
  'clinical_impression_2',
  'clinical_impression_3',
  'cli_impres_group',
  'cli_impres_2_group',
  'cli_impres_3_group',
  'melanoma_flag',
  'pathology_report',
  'biopsed',
  'background_father',
  'background_mother',
  'race',
  'race_group',
  'distance',
  'distance_group',
  'is_control',
  'diagnosis'],
 'source_input': 'combined_dataset_standardised.csv',
 'generated_files': {'stage1_csv': 'combined_stage1.csv',
  'stage2_csv': 

## Final verification

Checks:
- Reload exported CSVs
- Verify label balance
- Verify `sample_id` overlap (should be complete)


In [26]:
s1 = pd.read_csv(OUT_STAGE1)
s2 = pd.read_csv(OUT_STAGE2)

print("Reloaded Stage1:", s1.shape)
print("Reloaded Stage2:", s2.shape)

print("\nStage1 label balance:")
print(s1["y"].value_counts(dropna=False))

print("\nStage2 label balance:")
print(s2["y"].value_counts(dropna=False))

print("\nSample_id overlap (Stage1 ∩ Stage2):", len(set(s1["sample_id"]) & set(s2["sample_id"])))

Reloaded Stage1: (5648, 5)
Reloaded Stage2: (5659, 23)

Stage1 label balance:
y
1.0    3397
0.0    2251
Name: count, dtype: int64

Stage2 label balance:
y
1.0    3402
0.0    2257
Name: count, dtype: int64

Sample_id overlap (Stage1 ∩ Stage2): 5648
